# Aprenentatge per reforç

Els algorismes per reforç són una forma d'aprenentatge automàtic que permet a les màquines
(un agent) aprendre a prendre decisions de manera autònoma. Aquests algoritmes simulen un en-
torn en què l'agent pot prendre accions i rebre recompenses o càstigs en funció dels seus
resultats. A mesura que aquest pren més decisions/accions, aprèn quines li donen més recompenses i es torna més hàbil.

![Reforç Loop](./RL_loop.png)

1 El `Agent` observa el `State` del `environment`.
2 El `Agent` escull una `Action`.
3 El `Environment` reaccionar i dona al `Agent` un `Reward`  i un nou `state`
4 El `Agent` aprèn i el `Loop` es repeteix. 

Intuïtivament poden entendre que es va generant un dataset anomenat `Q-table` que li diu a l'agent com de bona és cada acció en cada situació. La diferència és que nosaltres no sabem generar el dataset complet i s'ha de generar conforme l'agent va provant. 

Per a no generar bucles a la simulació, l'agent ha d'anar canviant el seu comportament:

1 Provar una acció i veure què passa.
2 Actualitzar la `Q-Table` amb l'acció, estat i el reward.
3 Millorar gradualment provant accions i actualitzant els valors estimats.
4 Balancejar `exploration` vs `exploitation` (Provar coses noves vs utilitzar el que ja sap)
5 Al llarg del temps les bones accions tenen millors `Q-Values`, les males pitjors i l'agent aprèn a triar les accions amb majors `rewards` esperables. 

![Q-table](./qaction.png)




https://github.com/avidaldo/ia25/blob/master/qlearning/qlearning.ipynb
https://github.com/avidaldo/ia25/blob/master/qlearning/frozen_lake.ipynb
https://github.com/avidaldo/ia25/blob/master/qlearning/bellman_equation.ipynb


In [2]:


import numpy as np
import random
import time
from IPython.display import clear_output
     
# Environment parameters
GRID_ROWS = 4
GRID_COLS = 4
START_STATE = (0, 0)
GOAL_STATE = (3, 3)
TRAP_STATE = (2, 2)

# Rewards
REWARD_STEP = -0.1
REWARD_GOAL = 10
REWARD_TRAP = -10

# Actions (0: Up, 1: Down, 2: Left, 3: Right)
# We use indices for easier lookup in our Q-table
ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = ["↑", "↓", "←", "→"]

# A dictionary to map action indices to (row_change, col_change)
# This is how we'll move on the grid
ACTION_VECTORS = {
    0: (-1, 0), # Up
    1: (1, 0),  # Down
    2: (0, -1), # Left
    3: (0, 1)   # Right
}

print(f"Grid World: {GRID_ROWS}x{GRID_COLS}")
print(f"Start: {START_STATE}, Goal: {GOAL_STATE}, Trap: {TRAP_STATE}")
print(f"Actions: {list(zip(ACTIONS, ACTION_NAMES))}")

Grid World: 4x4
Start: (0, 0), Goal: (3, 3), Trap: (2, 2)
Actions: [(0, '↑'), (1, '↓'), (2, '←'), (3, '→')]


Hem creat un mon simple de 4x4. Es tracta de que l'agent es desplace pel mon amb 4 possibles accions. L'agent rebrà -0.1 per pas, per premiar els camins més curts. +10 per arribar al final, -10 per caure a la trampa. 

La Q-Table és com guardarà els seus coneixements l'agent. En aquest cas és una tabla 3D on es guarden les files, columnes i l'acció. Per tant, Q_table[row,col,action] guardarà un número, el `Q-Value`, que és la predicció de la recompensa si fa aquesta acció. 

In [3]:


Q_table = np.zeros((GRID_ROWS, GRID_COLS, len(ACTIONS)))

print("Initial Q-Table (all zeros):")
print(Q_table)
print(f"Shape of Q-Table: {Q_table.shape}")
     


Initial Q-Table (all zeros):
[[[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
Shape of Q-Table: (4, 4, 4)


Quan l'agent fa una acció d'un estat i es mou a un altre estat rep una recompensa i actualitzem la taula. S'utilitza la `Bellman Equation`:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

La idea és que s'actualitze el Q-Value de l'estat i l'acció sumant la nova informació per  el `learning rate` ($\alpha$). La nova informació és la recompensa `r` mes el màxim Q-value del nou estat, multiplicat per un factor de descompte ($\gamma$) menys el Q-Value actual. 


In [6]:
# Hyperparameters
learning_rate = 0.1   # Alpha (α): How quickly the agent learns.
discount_factor = 0.9 # Gamma (γ): How much the agent values future rewards.
epsilon = 1.0         # Initial exploration rate
max_epsilon = 1.0     # Maximum exploration
min_epsilon = 0.01    # Minimum exploration
epsilon_decay = 0.001 # Rate at which exploration decreases

# Training parameters
total_episodes = 10000 # How many "games" to play
max_steps = 100        # Max steps per game (to prevent infinite loops)

def choose_action(state, current_epsilon):
    """
    Chooses an action using the Epsilon-Greedy strategy.
    """
    row, col = state
    
    # Epsilon-Greedy decision
    if random.uniform(0, 1) < current_epsilon:
        # Explore: pick a random action
        return random.choice(ACTIONS)
    else:
        # Exploit: pick the best action from the Q-table
        # np.argmax finds the index (0, 1, 2, or 3) of the highest Q-value
        return np.argmax(Q_table[row, col])

def take_action(state, action_index):
    """
    Takes an action, calculates the new state, reward, and if the episode is done.
    """
    current_row, current_col = state
    action_row, action_col = ACTION_VECTORS[action_index]
    
    # Calculate new potential position
    new_row = current_row + action_row
    new_col = current_col + action_col
    
    # --- Check for wall collisions ---
    # Clamp the row to be within [0, GRID_ROWS - 1]
    new_row = max(0, min(new_row, GRID_ROWS - 1))
    # Clamp the col to be within [0, GRID_COLS - 1]
    new_col = max(0, min(new_col, GRID_COLS - 1))
    
    new_state = (new_row, new_col)
    
    # --- Get reward and check if done ---
    if new_state == GOAL_STATE:
        reward = REWARD_GOAL
        done = True
    elif new_state == TRAP_STATE:
        reward = REWARD_TRAP
        done = True
    else:
        reward = REWARD_STEP
        done = False
        
    return new_state, reward, done

In [ ]:
# To store rewards for plotting later
episode_rewards = []
current_epsilon = epsilon

for episode in range(total_episodes):
    state = START_STATE
    total_reward = 0
    
    for step in range(max_steps):
        # 1. Choose an action
        action_index = choose_action(state, current_epsilon)
        
        # 2. Take the action
        new_state, reward, done = take_action(state, action_index)
        
        # 3. Update the Q-table (The Q-Learning formula)
        row, col = state
        new_row, new_col = new_state
        
        old_q_value = Q_table[row, col, action_index]
        
        # This is max(Q(s', a')) from the formula
        best_future_q = np.max(Q_table[new_row, new_col])
        
        # The core Q-Learning update rule
        new_q_value = old_q_value + learning_rate * (reward + discount_factor * best_future_q - old_q_value)
        Q_table[row, col, action_index] = new_q_value
        
        # 4. Update state and reward
        state = new_state
        total_reward += reward
        
        if done:
            break # Episode finished
            
    # After the episode, store rewards and decay epsilon
    episode_rewards.append(total_reward)
    
    # Decay epsilon
    current_epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-epsilon_decay * episode)
    
    if (episode + 1) % 1000 == 0:
        print(f"Episode {episode + 1}/{total_episodes} | Epsilon: {current_epsilon:.4f}")


Episode 1000/10000 | Epsilon: 0.3746
Episode 2000/10000 | Epsilon: 0.1441
Episode 3000/10000 | Epsilon: 0.0593
Episode 4000/10000 | Epsilon: 0.0282
Episode 5000/10000 | Epsilon: 0.0167
Episode 6000/10000 | Epsilon: 0.0125
Episode 7000/10000 | Epsilon: 0.0109
Episode 8000/10000 | Epsilon: 0.0103
Episode 9000/10000 | Epsilon: 0.0101
Episode 10000/10000 | Epsilon: 0.0100
Final Q-Table:
 [[[  4.845851     5.49539      4.845851     5.49539   ]
  [  5.49539      6.2171       4.845851     6.2171    ]
  [  6.21709959   7.01899777   5.49538943   7.019     ]
  [  7.01899833   7.91         6.21709901   7.01899999]]

 [[  4.845851     6.2171       5.49539      6.2171    ]
  [  5.49539      7.019        5.49539      7.019     ]
  [  6.21709829  -9.99999995   6.21709818   7.91      ]
  [  7.01899971   8.9          7.01899991   7.90999998]]

 [[  5.49539      7.019        6.2171       7.019     ]
  [  6.2171       7.91         6.2171     -10.        ]
  [  0.           0.           0.           0.   

In [9]:

print("Final Q-Table:\n", Q_table)

# In[8]:
print("🎓 Learned Policy (Best action from each state):")

# Create a grid to store our policy arrows
policy_grid = [["" for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]

for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        state = (r, c)
        
        if state == GOAL_STATE:
            policy_grid[r][c] = "🏆" # Goal
        elif state == TRAP_STATE:
            policy_grid[r][c] = "🔥" # Trap
        else:
            # Find the best action (index) from this state
            best_action_index = np.argmax(Q_table[r, c])
            # Map that index to its arrow
            policy_grid[r][c] = ACTION_NAMES[best_action_index]

# Print the policy grid
for row in policy_grid:
    # .join(row) combines all elements in the list into a string
    # We use \t (a tab) to space them out nicely
    print("\t".join(row))

Final Q-Table:
 [[[  4.845851     5.49539      4.845851     5.49539   ]
  [  5.49539      6.2171       4.845851     6.2171    ]
  [  6.21709959   7.01899777   5.49538943   7.019     ]
  [  7.01899833   7.91         6.21709901   7.01899999]]

 [[  4.845851     6.2171       5.49539      6.2171    ]
  [  5.49539      7.019        5.49539      7.019     ]
  [  6.21709829  -9.99999995   6.21709818   7.91      ]
  [  7.01899971   8.9          7.01899991   7.90999998]]

 [[  5.49539      7.019        6.2171       7.019     ]
  [  6.2171       7.91         6.2171     -10.        ]
  [  0.           0.           0.           0.        ]
  [  7.90983342  10.          -9.9998971    8.89996919]]

 [[  6.21709992   7.01899979   7.01899998   7.91      ]
  [  7.019        7.91         7.019        8.9       ]
  [-10.           8.9          7.91        10.        ]
  [  0.           0.           0.           0.        ]]]
🎓 Learned Policy (Best action from each state):
↓	↓	→	↓
↓	↓	→	↓
↓	↓	🔥	↓
→	→	→	🏆


In [11]:
state = START_STATE
total_reward = 0
step_count = 0

for _ in range(max_steps):
    # Print the current grid
    clear_output(wait=True) # Clears the output for a nice animation
    
    # Create a temporary grid to print
    print_grid = [["." for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]
    print_grid[GOAL_STATE[0]][GOAL_STATE[1]] = "🏆"
    print_grid[TRAP_STATE[0]][TRAP_STATE[1]] = "🔥"
    print_grid[state[0]][state[1]] = "🤖" # Agent's current position
    
    print(f"Step: {step_count} | Total Reward: {total_reward:.1f}")
    for row in print_grid:
        print("\t".join(row))

    # --- Take the BEST action (no exploration) ---
    row, col = state
    action_index = np.argmax(Q_table[row, col])
    
    new_state, reward, done = take_action(state, action_index)
    
    state = new_state
    total_reward += reward
    step_count += 1
    
    time.sleep(0.5) # Pause for 0.5 seconds to see the move
    
    if done:
        # Print the final state
        clear_output(wait=True)
        print_grid = [["." for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]
        print_grid[GOAL_STATE[0]][GOAL_STATE[1]] = "🏆"
        print_grid[TRAP_STATE[0]][TRAP_STATE[1]] = "🔥"
        print_grid[state[0]][state[1]] = "🤖"
        
        print(f"Step: {step_count} | Total Reward: {total_reward:.1f}")
        for row in print_grid:
            print("\t".join(row))
        
        if state == GOAL_STATE:
            print("\n🎉 Agent reached the goal! 🎉")
        else:
            print("\n☠️ Agent fell in the trap! ☠️")
        break

Step: 6 | Total Reward: 9.5
.	.	.	.
.	.	.	.
.	.	🔥	.
.	.	.	🤖

🎉 Agent reached the goal! 🎉
